# Tutorial 5: MERFISH Aging multi-donor analysis

This tutorial describes a multi-section GraphPCA-Turbo analysis of the MERFISH Aging mouse-brain cohort (376,107 cells, 31 sections, 12 female donors at 4, 24, and 90 weeks; 374 measured genes). It focuses on interpretable shared programmes, donor-level transfer, and a carefully scoped age-specificity analysis. The full atlas and fitted embeddings are not bundled with the documentation.

## Goal

You will learn to prepare section objects, fit a shared-loading model, apply one common rotation for programme interpretation, map a programme in tissue, project a held-out donor, and quantify a target-cell-type contrast without treating it as proof of absolute activity.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

RUN_FULL_DATA = False
RAW_H5AD = Path('/path/to/MERFISH_Aging.h5ad')
RESULT_DIR = Path('/path/to/MERFISH_Aging_model')
SECTION_KEY = 'section'
DONOR_KEY = 'donor'
AGE_KEY = 'age_weeks'
CELLTYPE_KEY = 'cell_type'
SPATIAL_KEY = 'spatial'

print('Set RUN_FULL_DATA=True only after assigning the local atlas and output paths.')

## 1. Prepare the 31 section objects

The selected analysis used the source expression matrix after the established study preprocessing, with the same 374-gene panel in every section and an independently constructed 10-nearest-neighbour spatial graph per section. Do not concatenate sections or create cross-donor spatial edges. Preserve donor, age, cell-type annotation, and within-section coordinates as metadata for post hoc interpretation.

If reprocessing raw counts rather than reusing the validated source matrix, normalize within cells, apply `log1p`, and apply the same gene scaling to every section before fitting. Keep `center=True` below: GraphPCA-Turbo then centers each section internally.

In [ ]:
if RUN_FULL_DATA:
    import scanpy as sc
    from scipy.spatial import cKDTree
    from scipy import sparse

    adata = sc.read_h5ad(RAW_H5AD)
    required = {SECTION_KEY, DONOR_KEY, AGE_KEY, CELLTYPE_KEY}
    missing = required.difference(adata.obs.columns)
    if missing or SPATIAL_KEY not in adata.obsm:
        raise ValueError(f'Missing metadata={sorted(missing)} or obsm[{SPATIAL_KEY!r}].')
    sections = []
    locations = []
    for section_name in adata.obs[SECTION_KEY].astype(str).drop_duplicates():
        one = adata[adata.obs[SECTION_KEY].astype(str).eq(section_name)].copy()
        sections.append(one)
        locations.append(np.asarray(one.obsm[SPATIAL_KEY])[:, :2])
    assert len(sections) == 31
    assert len(set(map(tuple, [x.var_names for x in sections]))) == 1
    print(len(sections), sections[0].shape)
else:
    print('Full-data read skipped.')

## 2. Learn a comparable shared programme basis

The global centre `W0` is the shared gene-loading basis; each `Ws` retains section-specific refinement. `Zs` are spatially regularized embeddings. This in-memory fit is appropriate for the 376k-cell cohort; use the disk-backed mode in the MOSTA tutorial when the full cohort does not fit comfortably in memory.

In [ ]:
if RUN_FULL_DATA:
    from GraphPCA import Run_Hierarchical_Multi_GPCA
    Zs, W0, Ws, info = Run_Hierarchical_Multi_GPCA(
        adatas=sections, locations=locations, n_components=30,
        lambdas=0.5, rhos=0.15, sample_weights='equal_slice',
        center=True, mode='accelerated', pcg_tol=1e-6, pcg_max_iter=500,
        max_iter=50, random_seed=666, return_info=True,
    )
    print(info.converged, info.n_iter, W0.shape, len(Zs))
else:
    print('Full fit skipped.')

## 3. Interpret shared programmes

Components have an orthogonal rotation/sign ambiguity. Rotate `W0` once, orient each column deterministically, and apply the same rotation to every `Ws` and `Zs`. Then a programme index has the same molecular definition across sections. Leading genes and spatial score maps are complementary evidence; neither alone establishes cell-state activity.

In [ ]:
def common_varimax(loadings, n_iter=100, tol=1e-6):
    rotation = np.eye(loadings.shape[1])
    for _ in range(n_iter):
        rotated = loadings @ rotation
        u, _, vt = np.linalg.svd(loadings.T @ (rotated**3 - rotated @ np.diag(np.mean(rotated**2, axis=0))))
        updated = u @ vt
        if np.linalg.norm(updated - rotation) < tol:
            break
        rotation = updated
    return rotation

if RUN_FULL_DATA:
    rotation = common_varimax(W0)
    W0_rot = W0 @ rotation
    Zs_rot = [z @ rotation for z in Zs]
    component = 7  # zero-based P08 after the saved reporting order is established
    gene_table = pd.DataFrame({'gene': sections[0].var_names, 'loading': W0_rot[:, component]})
    print(gene_table.nlargest(10, 'loading'))
    section_index = 0
    xy = locations[section_index]
    plt.scatter(xy[:,0], xy[:,1], c=Zs_rot[section_index][:, component], s=1, cmap='coolwarm', linewidths=0)
    plt.gca().set_aspect('equal'); plt.colorbar(label='rotated P08 score'); plt.show()

## 4. Donor-held-out projection and age specificity

For donor-held-out transfer, learn `W0` using only the remaining donors and project every section of the held-out donor with the fixed loading basis. Compare the resulting representation with the corresponding full-data reference only after aligning component coordinates.

For the supported P08 endothelial/BBB example, compute the donor-level difference between endothelial cells and all other cells, then standardize that *relative specificity* contrast to the young-donor reference. This is not a claim of absolute programme activation and should not be generalized to the retired P24 interpretation.

In [ ]:
if RUN_FULL_DATA:
    from GraphPCA import Project_Hierarchical_Multi_GPCA
    held_out_donor = str(sections[0].obs[DONOR_KEY].iloc[0])
    train_index = [i for i, one in enumerate(sections) if str(one.obs[DONOR_KEY].iloc[0]) != held_out_donor]
    # Fit the same model on [sections[i] for i in train_index] to obtain W0_train.
    # For each held-out section: Z_new, W_new = Project_Hierarchical_Multi_GPCA(
    #     adata=one, global_loading=W0_train, location=one.obsm[SPATIAL_KEY], graph_lambda=0.5, rho=0.15)
    
    scores = pd.DataFrame({
        'donor': sections[0].obs[DONOR_KEY].to_numpy(),
        'age_weeks': sections[0].obs[AGE_KEY].to_numpy(),
        'cell_type': sections[0].obs[CELLTYPE_KEY].to_numpy(),
        'P08': Zs_rot[0][:, 7],
    })
    def relative_specificity(frame, target='Endothelial'):
        target_mean = frame.loc[frame.cell_type.eq(target), 'P08'].mean()
        rest_mean = frame.loc[~frame.cell_type.eq(target), 'P08'].mean()
        return target_mean - rest_mean
    donor_contrast = scores.groupby(['donor', 'age_weeks']).apply(relative_specificity).rename('P08_relative_specificity').reset_index()
    young = donor_contrast.loc[donor_contrast.age_weeks.eq(4), 'P08_relative_specificity']
    donor_contrast['young_reference_z'] = (donor_contrast.P08_relative_specificity - young.mean()) / young.std(ddof=1)
    print(donor_contrast)
else:
    print('Projection and age-specificity calculations require the local model outputs.')

## Checks and next steps

Record the input gene order, spatial graph definition, `rho`, rotation matrix, donor split, and component orientation with the model output. Use held-out donor prediction and section-level spatial maps as checks of transfer and interpretation. Do not report unrun pseudotime/trajectory results, and treat donor-level programme specificity as a relative contrast rather than a causal or absolute-activity measurement.